# Extract family corpus (`extracted_families.json`)

This notebook is the **producer** of `data/extracted_families.json`.
The runtime consumer is `CorpusRepository` (and the copy packaged in the wheel).

- **Input:** `data/ApexChem_Synthesis_Reactions_by_AminoAcid.xlsx`
- **Output schema:** `schema_version` 2.0.0, families with `processes` keyed by `process_id`
- **Checkpoint:** `data/extracted_families.json`. Re-run only when the workbook or extraction rules change.

It is not a conflict matrix. Walker conflicts are computed at runtime from family bindings.


In [ ]:
from __future__ import annotations

import json
import re
import os
from dataclasses import dataclass
from enum import StrEnum
from pathlib import Path
from typing import Any, Literal

from openpyxl import load_workbook
from openpyxl.workbook.workbook import Workbook
from openpyxl.worksheet.worksheet import Worksheet
from pydantic import BaseModel, ConfigDict, Field, model_validator


class Family(StrEnum):
    SPPS_FOUNDATION = "spps_foundation"
    SPECIAL_RESIDUES = "special_residues"
    N_METHYLATION = "n_methylation"
    C_TERM_AMIDATION = "c_term_amidation"
    N_TERM_ACETYLATION = "n_term_acetylation"
    LIPIDATION = "lipidation"
    PEGYLATION = "pegylation"
    GLYCOSYLATION = "glycosylation"
    CYCLIZATION = "cyclization"
    HYDROCARBON_STAPLING = "hydrocarbon_stapling"
    DISULFIDE = "disulfide"
    BIARYL_BISALKYLATION = "biaryl_bisalkylation"
    AZA_PEPTIDE = "aza_peptide"
    RETRO_INVERSO = "retro_inverso"
    CHARGE_HYBRIDS = "charge_hybrids"


class ConflictKind(StrEnum):
    PROTECTING_GROUP_ORTHOGONALITY = "protecting_group_orthogonality"
    ORDER_OF_OPERATIONS = "order_of_operations"
    MUTUALLY_EXCLUSIVE = "mutually_exclusive"
    REAGENT_INCOMPATIBILITY = "reagent_incompatibility"
    BUILDING_BLOCK_AVAILABILITY = "building_block_availability"
    INTENT_NOT_ACHIEVED = "intent_not_achieved"


class Severity(StrEnum):
    BLOCKING = "blocking"
    MAJOR = "major"
    MINOR = "minor"


class MaterialKind(StrEnum):
    REQUIRES = "requires"
    REAGENT = "reagent"
    CONDITION = "condition"
    PROTECTING_GROUP = "protecting_group"
    CONSTRAINT = "constraint"
    CAVEAT = "caveat"


## Setup

The notebook has to run from the repository root or from the notebooks directory. find_repo_root walks upward until it sees both pyproject.toml and a notebooks folder, then every later class receives a Path.

In [ ]:
def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "notebooks").is_dir():
            return candidate
    raise RuntimeError("Launch the notebook from the repository root or from notebooks/.")


REPO_ROOT = find_repo_root()
DATA_DIR = REPO_ROOT / "data"
WORKBOOK_PATH = DATA_DIR / "ApexChem_Synthesis_Reactions_by_AminoAcid.xlsx"
CHECKPOINT_PATH = DATA_DIR / "extracted_families.json"
SCHEMA_VERSION = "2.0.0"


## Step 1. Open the workbook and bind each Family to one sheet

CorpusWorkbook is the only object that talks to openpyxl. It receives the file path in __init__, loads the workbook once, and builds a Family.


In [ ]:
@dataclass(frozen=True)
class CorpusRef:
    kind: Literal["corpus"] = "corpus"
    ref: str = ""

    def as_dict(self) -> dict[str, str]:
        return {"kind": self.kind, "ref": self.ref}


class CorpusWorkbook:
    # Workbook plus Family-to-sheet index. No module-level client.

    def __init__(self, path: Path) -> None:
        if not path.is_file():
            raise FileNotFoundError(f"Missing workbook: {path}")
        self.path = path
        self.stem = path.stem
        self._workbook: Workbook = load_workbook(path, data_only=True)
        self._sheets = self._map_family_sheets()

    def _map_family_sheets(self) -> dict[Family, str]:
        mapping: dict[Family, str] = {}
        for index, family in enumerate(Family, start=1):
            prefix = f"{index:02d}_"
            matches = [name for name in self._workbook.sheetnames if name.startswith(prefix)]
            if len(matches) != 1:
                raise ValueError(
                    f"Expected one sheet {prefix!r} for {family.value}; found {matches}"
                )
            mapping[family] = matches[0]
        return mapping

    def sheet_name(self, family: Family) -> str:
        return self._sheets[family]

    def worksheet(self, family: Family) -> Worksheet:
        return self._workbook[self.sheet_name(family)]

    def worksheet_by_title(self, title: str) -> Worksheet:
        return self._workbook[title]

    def cite(self, sheet: str, row: int) -> CorpusRef:
        return CorpusRef(ref=f"{self.stem}:{sheet}:{row}")

    @staticmethod
    def cell_text(value: Any) -> str:
        return "" if value is None else str(value).strip()

    def row_values(self, ws: Worksheet, row: int) -> list[str]:
        return [self.cell_text(cell.value) for cell in ws[row] if self.cell_text(cell.value)]

    def header_row(self, ws: Worksheet) -> int:
        for row in range(1, min(ws.max_row, 12) + 1):
            values = self.row_values(ws, row)
            if values and values[0] == "Step":
                return row
        raise ValueError(f"{ws.title}: no header row starting with 'Step'")

    def header_columns(self, ws: Worksheet) -> dict[str, int]:
        row = self.header_row(ws)
        columns: dict[str, int] = {}
        for cell in ws[row]:
            header = self.cell_text(cell.value)
            if header and header not in columns:
                columns[header] = cell.column
        return columns

    def is_content_row(self, ws: Worksheet, row: int) -> bool:
        return row > self.header_row(ws) and row <= ws.max_row and bool(self.row_values(ws, row))

    def require_content_row(self, ws: Worksheet, row: int) -> None:
        if not self.is_content_row(ws, row):
            raise ValueError(f"{ws.title} row {row} is not a content row")

    def content_rows(self, family: Family) -> list[dict[str, Any]]:
        ws = self.worksheet(family)
        headers = self.header_columns(ws)
        rows: list[dict[str, Any]] = []
        for row in range(self.header_row(ws) + 1, ws.max_row + 1):
            if not self.row_values(ws, row):
                continue
            cells = {
                header: self.cell_text(ws.cell(row=row, column=column).value)
                for header, column in headers.items()
                if self.cell_text(ws.cell(row=row, column=column).value)
            }
            rows.append({"row": row, "cells": cells})
        return rows

    def named_cells(self, ws: Worksheet, row: int) -> list[dict[str, str]]:
        headers_by_index = {index: name for name, index in self.header_columns(ws).items()}
        cells: list[dict[str, str]] = []
        for cell in ws[row]:
            value = self.cell_text(cell.value)
            if not value:
                continue
            cells.append(
                {
                    "column": headers_by_index.get(cell.column, cell.column_letter),
                    "value": value,
                }
            )
        return cells

    def excerpt(self, sheet: str, row: int) -> str:
        return " | ".join(self.row_values(self.worksheet_by_title(sheet), row))


The next cell constructs one CorpusWorkbook and prints the fifteen bindings. Read that list before Phase A. If the printed sheet names do not match the family you expect, stop. A swapped tab would send the wrong chemistry into the checkpoint and every later citation would look locally valid while pointing at the wrong family.


In [ ]:
corpus = CorpusWorkbook(WORKBOOK_PATH)
print(f"total sheets: {len(corpus._workbook.sheetnames)}")
print("Family enum -> numbered tab")
for family in Family:
    print(f"  {family.value:24s}  {corpus.sheet_name(family)}")


## First phase

Phase A exists because the fifteen family tabs are too large to transcribe by hand and too easy to paraphrase incorrectly.

A **process** is a chemically coherent route on the tab: different reagents, different orthogonal handles, or different consequences/risks mean different processes. Parentheses in the Step column (`1 (HEAD-TO-TAIL)`, `2a (ON-RESIN alt)`) are a strong signal when present, but they are not the only signal — read the whole sheet (steps, reagents, conditions, risks, alternatives). Example: lipidation's Lys(Mtt) / Lys(ivDde) / Lys(Alloc) paths use different deprotection reagents and carry different Trt risks, so they are separate processes even when Step is just `1`, `2`, `3`.

Extraction is two-pass:

1. one LiteLLM call reads the sheet and discovers the distinct processes (label + which reaction-step rows anchor each);
2. one LiteLLM call per discovered process extracts that complete route: steps, reagents, conditions, protecting groups, requires, constraints, caveats, plus synthesized `summary` / `when_to_use`.

Python recopies every chemical string from openpyxl and rejects a process with no canonical steps.


In [ ]:
TYPE_DISCOVERY_PROMPT = """\
You discover distinct synthetic processes on one ApexChem family tab by reading \
the full sheet. A process is a chemically coherent route: if different reagents, \
orthogonal handles, or consequences/risks apply depending on the choice, those \
are separate processes — not one merged route.

Family: {family}
Sheet: {sheet}

Content rows (row number + non-empty cells). Use these exact row numbers.
{rows}

How to decide what counts as a distinct process
- Read Step, What happens, Reagents, Conditions, RISKS / QC, and ALTERNATIVES \
together. Do not look only at parentheses.
- Step parentheses are a strong signal when present. Examples: \
"1 (HEAD-TO-TAIL)" and "3 (SIDE-CHAIN LACTAM)" are two processes; \
"1 (ON-RESIN)" vs "2 (SOLUTION)" are two processes; \
"1 (PREFERRED)" vs "2a (ON-RESIN alt)" are two processes.
- Also split when the sheet offers alternative chemistries with different \
reagents or different consequences, even if Step is only 1/2/3 with no \
parentheses. Example: lipidation lists Lys(Mtt) / Lys(ivDde) / Lys(Alloc) \
with 1% TFA/DCM vs 2% hydrazine/DMF vs Pd/PhSiH3, and a MAJOR TRAP that \
applies to Mtt/TFA versus Trt — emit one process per handle.
- Shared upstream or downstream steps may appear in more than one process's \
step_rows when each route needs them to stand alone.
- If the tab truly has one reagent path and one consequence profile, return \
exactly one process (label can be "default" or a short route name).
- Ignore banner-only rows as process labels (REAGENTS NAMED..., CONDITIONS, \
RISKS / QC, ALTERNATIVES, SYMPHONYX vs MANUAL, REVIEWER NOTES, lone "#"), but \
do use their chemistry text as evidence when deciding splits.
- Do not invent row numbers. Do not decide family-vs-family conflicts.

Return a ProcessTypeDiscovery:
- family_summary: one short sentence on how the processes on this tab differ \
(reagents and/or consequences), or that there is a single route
- types: one or more ProcessType objects, each with:
  - label: short process label (parenthetical Step label when that is the split, \
or a handle/reagent label such as Mtt / ivDde / Alloc when that is the split)
  - step_rows: reaction-step content row numbers that anchor this process
"""

PROCESS_EXTRACTION_PROMPT = """\
You extract ONE complete synthetic process from an ApexChem family tab. A later \
Python pass recopies exact cell text with openpyxl. You never invent chemical \
wording. You never decide whether two families conflict.

Family: {family}
Sheet: {sheet}
Process type label: {type_label}
Reaction-step rows that belong to THIS type (must drive the ordered steps):
{type_step_rows}

Named columns on this sheet. Use these header strings exactly when you fill columns:
{headers}

Content rows you may cite. Each item is a row number plus the non-empty cells \
on that row. Header rows are not in this list and must not be selected.
{rows}

Return a ProcessSelection for only this type, with:
- name: short route label that includes the type
- summary: what this route does
- when_to_use: when to prefer this route over the family's other types
- steps: ordered synthetic operations for this complete route. Prefer the \
type_step_rows listed above for operations. Shared upstream/downstream rows may \
be included only if this type needs them to stand alone.
- materials: supporting reagents, conditions, protecting groups, requirements, \
constraints, and caveats for THIS type (reagent-table rows, CONDITIONS, RISKS / QC, \
ALTERNATIVES, etc.). Assign a caveat to this type only when it applies to this \
type's chemistry.

Each step has:
- order: positive integer, unique within the process
- operation: CellSelection (ref_row + columns)
- materials: MaterialSelection items for reagents/conditions on that step (may cite \
supporting rows)

Each MaterialSelection has:
- kind: one of requires, reagent, condition, protecting_group, constraint, caveat
- selection: CellSelection (ref_row + columns)

What not to do
Do not paraphrase chemical cell text. Do not invent row numbers or column names. \
Stay on process {type_label}: keep its reagents and consequence profile; do not \
merge a competing alternative into this extraction. Prefer chemistry-bearing columns \
(What happens (from deck), Reagents named in deck, Conditions from deck, Notes) \
and ignore CAS, supplier, pack size, lead time, and in-stock unless they are the \
only populated cells on an otherwise chemical row.
"""


In [ ]:
class CellSelection(BaseModel):
    model_config = ConfigDict(extra="forbid")
    ref_row: int = Field(ge=1)
    columns: list[str] = Field(min_length=1)


class MaterialSelection(BaseModel):
    model_config = ConfigDict(extra="forbid")
    kind: MaterialKind
    selection: CellSelection


class StepSelection(BaseModel):
    model_config = ConfigDict(extra="forbid")
    order: int = Field(ge=1)
    operation: CellSelection
    # Materials may cite supporting rows (e.g. reagent table) as well as the
    # operation row; canonicalize copies whatever cells are selected.
    materials: list[MaterialSelection] = Field(default_factory=list)


class ProcessSelection(BaseModel):
    model_config = ConfigDict(extra="forbid")
    name: str = Field(min_length=1)
    summary: str = Field(min_length=1)
    when_to_use: str = Field(min_length=1)
    steps: list[StepSelection] = Field(min_length=1)
    materials: list[MaterialSelection] = Field(default_factory=list)

    @model_validator(mode="after")
    def _unique_step_orders(self) -> ProcessSelection:
        orders = [step.order for step in self.steps]
        if len(orders) != len(set(orders)):
            raise ValueError("step orders must be unique within a process")
        return self


class ProcessType(BaseModel):
    model_config = ConfigDict(extra="forbid")
    label: str = Field(min_length=1)
    step_rows: list[int] = Field(min_length=1)


class ProcessTypeDiscovery(BaseModel):
    model_config = ConfigDict(extra="forbid")
    family_summary: str = Field(min_length=1)
    types: list[ProcessType] = Field(min_length=1)


_SLUG_RE = re.compile(r"[^a-z0-9]+")


def slugify_process_name(name: str) -> str:
    slug = _SLUG_RE.sub("_", name.strip().lower()).strip("_")
    return slug or "process"


def assign_process_ids(processes: list[dict[str, Any]]) -> list[dict[str, Any]]:
    """Assign unique process_id values after sorting by first cited row."""

    def first_row(process: dict[str, Any]) -> int:
        rows: list[int] = []
        for step in process.get("steps", []):
            if "ref_row" in step:
                rows.append(int(step["ref_row"]))
        for item in process.get("provenance", []):
            if "ref_row" in item:
                rows.append(int(item["ref_row"]))
                continue
            ref = item.get("ref", "")
            if isinstance(ref, str) and ":" in ref:
                try:
                    rows.append(int(ref.rsplit(":", 1)[-1]))
                except ValueError:
                    continue
        return min(rows) if rows else 10**9

    ordered = sorted(processes, key=lambda item: (first_row(item), item["name"].lower()))
    counts: dict[str, int] = {}
    used: dict[str, int] = {}
    for process in ordered:
        base = slugify_process_name(process["name"])
        counts[base] = counts.get(base, 0) + 1
    for process in ordered:
        base = slugify_process_name(process["name"])
        if counts[base] == 1:
            process["process_id"] = base
            continue
        used[base] = used.get(base, 0) + 1
        process["process_id"] = f"{base}_{used[base]}"
    return ordered


def index_family_processes(
    families: dict[str, dict[str, Any]],
) -> dict[str, dict[str, Any]]:
    """Store processes as a process_id map, the runtime catalog shape."""
    indexed: dict[str, dict[str, Any]] = {}
    for family_name, profile in families.items():
        processes = profile["processes"]
        if isinstance(processes, list):
            processes = {item["process_id"]: item for item in processes}
        else:
            processes = dict(processes)
        indexed[family_name] = {
            "sheet": profile["sheet"],
            "summary": profile.get("summary", ""),
            "processes": processes,
        }
    return indexed


def sort_canonical_entries(entries: list[dict[str, Any]]) -> list[dict[str, Any]]:
    return sorted(
        entries,
        key=lambda item: (
            item["ref_row"],
            item.get("source_excerpt", ""),
            json.dumps(item.get("cells", []), sort_keys=True),
        ),
    )


def dedupe_canonical_entries(entries: list[dict[str, Any]]) -> list[dict[str, Any]]:
    seen: set[tuple[Any, ...]] = set()
    unique: list[dict[str, Any]] = []
    for entry in sort_canonical_entries(entries):
        key = (
            entry["ref_row"],
            entry["source_excerpt"],
            json.dumps(entry["cells"], sort_keys=True),
        )
        if key in seen:
            continue
        seen.add(key)
        unique.append(entry)
    return unique


def require_process_checkpoint(checkpoint: dict[str, Any], expected_version: str) -> None:
    version = checkpoint.get("schema_version")
    if version != expected_version:
        raise ValueError(
            f"Checkpoint schema_version={version!r} is stale; expected "
            f"{expected_version!r}. Set REFRESH_EXTRACTION = True to rebuild "
            "process-oriented extracted_families.json."
        )
    families = checkpoint.get("families")
    if not isinstance(families, dict) or not families:
        raise ValueError("Checkpoint families object is missing or empty")
    for name, profile in families.items():
        processes = profile.get("processes")
        if not isinstance(processes, dict) or not processes:
            raise ValueError(
                f"Family {name!r} must have a non-empty processes object "
                "keyed by process_id. Set REFRESH_EXTRACTION = True."
            )


class FamilyExtractor:
    """Two-pass extraction: discover Step-column types, then one call per type."""

    def __init__(
        self,
        corpus: CorpusWorkbook,
        model: str,
        type_prompt_template: str,
        process_prompt_template: str,
    ) -> None:
        self.corpus = corpus
        self.model = model
        self.type_prompt_template = type_prompt_template
        self.process_prompt_template = process_prompt_template

    def _completion(self, prompt: str, response_format: type[BaseModel]) -> Any:
        from litellm import completion

        response = completion(
            model=self.model,
            messages=[{"role": "user", "content": prompt}],
            response_format=response_format,
            temperature=0,
        )
        message = response.choices[0].message
        parsed = getattr(message, "parsed", None)
        if isinstance(parsed, response_format):
            return parsed
        raw = parsed if parsed is not None else message.content
        return response_format.model_validate(
            raw if isinstance(raw, dict) else json.loads(raw)
        )

    def _render_type_prompt(self, family: Family) -> str:
        ws = self.corpus.worksheet(family)
        return self.type_prompt_template.format(
            family=family.value,
            sheet=ws.title,
            rows=json.dumps(
                self.corpus.content_rows(family), ensure_ascii=False, indent=2
            ),
        )

    def _render_process_prompt(
        self, family: Family, process_type: ProcessType
    ) -> str:
        ws = self.corpus.worksheet(family)
        return self.process_prompt_template.format(
            family=family.value,
            sheet=ws.title,
            type_label=process_type.label,
            type_step_rows=json.dumps(process_type.step_rows),
            headers="\n".join(
                f"- {name}" for name in self.corpus.header_columns(ws)
            ),
            rows=json.dumps(
                self.corpus.content_rows(family), ensure_ascii=False, indent=2
            ),
        )

    def discover_types(self, family: Family) -> ProcessTypeDiscovery:
        discovery = self._completion(
            self._render_type_prompt(family), ProcessTypeDiscovery
        )
        ws = self.corpus.worksheet(family)
        validated: list[ProcessType] = []
        for process_type in discovery.types:
            rows: list[int] = []
            for row in process_type.step_rows:
                if not self.corpus.is_content_row(ws, row):
                    print(
                        f"  skip {ws.title} type {process_type.label!r} "
                        f"row {row}: not a content row"
                    )
                    continue
                rows.append(row)
            if not rows:
                print(f"  skip {ws.title} type {process_type.label!r}: no valid rows")
                continue
            validated.append(
                ProcessType(label=process_type.label.strip(), step_rows=sorted(set(rows)))
            )
        if not validated:
            raise ValueError(f"{family.value}: type discovery produced zero types")
        return ProcessTypeDiscovery(
            family_summary=discovery.family_summary.strip(),
            types=validated,
        )

    def _cells_from_selection(
        self, family: Family, selection: CellSelection
    ) -> list[dict[str, str]] | None:
        ws = self.corpus.worksheet(family)
        self.corpus.require_content_row(ws, selection.ref_row)
        headers = self.corpus.header_columns(ws)
        cells: list[dict[str, str]] = []
        for column in selection.columns:
            index = headers.get(column)
            if index is None:
                continue
            value = self.corpus.cell_text(ws.cell(selection.ref_row, index).value)
            if value:
                cells.append({"column": column, "value": value})
        if not cells:
            cells = self.corpus.named_cells(ws, selection.ref_row)
        return cells or None

    def _canonical_entry(
        self, family: Family, selection: CellSelection
    ) -> dict[str, Any] | None:
        cells = self._cells_from_selection(family, selection)
        if cells is None:
            return None
        ws = self.corpus.worksheet(family)
        return {
            "ref_row": selection.ref_row,
            "cells": cells,
            "source_excerpt": " | ".join(cell["value"] for cell in cells),
            "provenance": [self.corpus.cite(ws.title, selection.ref_row).as_dict()],
        }

    def _canonical_material(
        self, family: Family, material: MaterialSelection
    ) -> tuple[str, dict[str, Any]] | None:
        entry = self._canonical_entry(family, material.selection)
        if entry is None:
            return None
        return material.kind.value, entry

    def _canonicalize_process(
        self, family: Family, process: ProcessSelection, type_label: str
    ) -> dict[str, Any] | None:
        inventory: dict[str, list[dict[str, Any]]] = {
            kind.value: [] for kind in MaterialKind
        }
        steps: list[dict[str, Any]] = []
        provenance_rows: dict[int, dict[str, str]] = {}

        def remember(entry: dict[str, Any]) -> None:
            for item in entry["provenance"]:
                provenance_rows[entry["ref_row"]] = item

        for step in sorted(process.steps, key=lambda item: item.order):
            operation = self._canonical_entry(family, step.operation)
            if operation is None:
                print(
                    f"  skip {self.corpus.sheet_name(family)} process "
                    f"{process.name!r} step {step.order}: empty operation"
                )
                continue
            remember(operation)
            step_materials: dict[str, list[dict[str, Any]]] = {
                "requires": [],
                "reagents": [],
                "conditions": [],
            }
            for material in step.materials:
                canonical = self._canonical_material(family, material)
                if canonical is None:
                    continue
                kind, entry = canonical
                remember(entry)
                inventory[kind].append(entry)
                if kind == MaterialKind.REQUIRES.value:
                    step_materials["requires"].append(entry)
                elif kind == MaterialKind.REAGENT.value:
                    step_materials["reagents"].append(entry)
                elif kind == MaterialKind.CONDITION.value:
                    step_materials["conditions"].append(entry)
            steps.append(
                {
                    "order": step.order,
                    "operation": operation["source_excerpt"],
                    "ref_row": operation["ref_row"],
                    "requires": dedupe_canonical_entries(step_materials["requires"]),
                    "reagents": dedupe_canonical_entries(step_materials["reagents"]),
                    "conditions": dedupe_canonical_entries(step_materials["conditions"]),
                    "provenance": operation["provenance"],
                }
            )

        if not steps:
            return None

        for material in process.materials:
            canonical = self._canonical_material(family, material)
            if canonical is None:
                continue
            kind, entry = canonical
            remember(entry)
            inventory[kind].append(entry)

        payload = {
            "type_label": type_label,
            "name": process.name.strip(),
            "summary": process.summary.strip(),
            "when_to_use": process.when_to_use.strip(),
            "requires": dedupe_canonical_entries(inventory[MaterialKind.REQUIRES.value]),
            "reagents": dedupe_canonical_entries(inventory[MaterialKind.REAGENT.value]),
            "conditions": dedupe_canonical_entries(
                inventory[MaterialKind.CONDITION.value]
            ),
            "protecting_groups": dedupe_canonical_entries(
                inventory[MaterialKind.PROTECTING_GROUP.value]
            ),
            "constraints": dedupe_canonical_entries(
                inventory[MaterialKind.CONSTRAINT.value]
            ),
            "caveats": dedupe_canonical_entries(inventory[MaterialKind.CAVEAT.value]),
            "steps": steps,
            "provenance": [provenance_rows[row] for row in sorted(provenance_rows)],
        }
        if not payload["name"] or not payload["summary"] or not payload["when_to_use"]:
            return None
        if not payload["provenance"]:
            return None
        return payload

    def extract_process(
        self, family: Family, process_type: ProcessType
    ) -> dict[str, Any] | None:
        selection = self._completion(
            self._render_process_prompt(family, process_type), ProcessSelection
        )
        return self._canonicalize_process(family, selection, process_type.label)

    def extract(self, family: Family) -> dict[str, Any]:
        ws = self.corpus.worksheet(family)
        discovery = self.discover_types(family)
        print(
            f"  types for {family.value}: "
            + ", ".join(
                f"{item.label}{item.step_rows}" for item in discovery.types
            )
        )
        processes: list[dict[str, Any]] = []
        for process_type in discovery.types:
            print(f"  extracting type {process_type.label!r}")
            canonical = self.extract_process(family, process_type)
            if canonical is None:
                print(
                    f"  skip {ws.title} type {process_type.label!r}: no canonical steps"
                )
                continue
            processes.append(canonical)
        if not processes:
            raise ValueError(f"{family.value}: extraction produced zero valid processes")
        processes = assign_process_ids(processes)
        return {
            "sheet": ws.title,
            "summary": discovery.family_summary,
            "processes": {item["process_id"]: item for item in processes},
        }

    def extract_families(self, families: list[Family]) -> dict[str, dict[str, Any]]:
        result: dict[str, dict[str, Any]] = {}
        for family in families:
            print(f"extracting {family.value} ({self.corpus.sheet_name(family)})")
            result[family.value] = self.extract(family)
        return result


In [ ]:
ROUTE_AGENT_MODEL = os.environ.get("ROUTE_AGENT_MODEL", "openai/gpt-4o-mini")


def read_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding="utf-8"))


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.write_text(
        json.dumps(payload, indent=2, ensure_ascii=False) + "\n",
        encoding="utf-8",
    )


def extracted_families_document(
    *,
    model: str,
    schema_version: str,
    source_workbook: str,
    family_order: list[str],
    families: dict[str, dict[str, Any]],
) -> dict[str, Any]:
    return {
        "model": model,
        "schema_version": schema_version,
        "source_workbook": source_workbook,
        "family_order": family_order,
        "families": index_family_processes(families),
    }


### Run the extraction or reuse the checkpoint

This is the only cell that may call LiteLLM. It writes `data/extracted_families.json` in the runtime shape: top-level `model` / `schema_version` / `source_workbook` / `family_order`, and each family's `processes` keyed by `process_id`. There is no separate reindex step.


In [ ]:
provider = ROUTE_AGENT_MODEL.split("/", 1)[0]
env_name = {"openai": "OPENAI_API_KEY", "anthropic": "ANTHROPIC_API_KEY"}.get(provider)
if env_name is None:
    raise RuntimeError(f"Unknown LiteLLM provider in {ROUTE_AGENT_MODEL!r}")
if not os.environ.get(env_name):
    raise RuntimeError(f"REFRESH_EXTRACTION=True requires {env_name}")
    
extractor = FamilyExtractor(
    corpus,
    ROUTE_AGENT_MODEL,
    TYPE_DISCOVERY_PROMPT,
    PROCESS_EXTRACTION_PROMPT,
)

families = extractor.extract_families(list(Family))
checkpoint = extracted_families_document(
    model=ROUTE_AGENT_MODEL,
    schema_version=SCHEMA_VERSION,
    source_workbook=corpus.stem,
    family_order=[family.value for family in Family],
    families=families,
)
write_json(CHECKPOINT_PATH, checkpoint)
require_process_checkpoint(checkpoint, SCHEMA_VERSION)
print(f"wrote {CHECKPOINT_PATH}")
print(f"Phase A extracted {len(checkpoint['families'])} families with {ROUTE_AGENT_MODEL}")
for name, profile in list(checkpoint["families"].items())[:3]:
    print(f"  {name}: {list(profile['processes'])}")